# Taxi-v4 as a finite MDP: extracting the model, component by component

**Goal.** Extract the full MDP $(\mathcal S,\mathcal A,p,r,\gamma,\mu_0)$ of Gymnasium's `Taxi-v4`, where:
- $\mathcal S$ is the set of states and $\mathcal A$ the set of actions;
- $p(s'\mid s,a)$ is the probability of moving to $s'$ after taking $a$ in $s$;
- $r(s,a)$ is the expected immediate reward. Zhao writes the model as the pair $p(s'\mid s,a)$, $p(r\mid s,a)$; Section 5.1 shows how $p(r\mid s,a)$ reduces to $r(s,a)$ in Taxi;
- $\gamma\in[0,1)$ is the **discount factor**: a reward received $k$ steps in the future is weighted by $\gamma^k$, so the return is $G_0=\sum_{k\ge0}\gamma^k R_{k+1}$.
- $\mu_0$ is the distribution of the initial state.
Every markdown cell first states the **expected result**, computed by hand.
The code cell after it **checks** that result with an `assert`.

**Plan**
1. Setup and the ANSI render
2. State space $\mathcal S$ (encoding, start states, reachable states)
3. Action space $\mathcal A$
4. Transition model $p(s'\mid s,a)$: the raw `P` dictionary, then tensors
5. Rewards $r(s,a)$
6. Termination, and the start distribution $\mu_0$
7. Rainy dynamics (stochastic $p$)
8. Manual interaction: playing an episode by hand
9. Why `fickle_passenger` is not in the model
10. Summary

Running example, the state you got from your first `reset()`: the taxi is empty at $(2,2)$, the passenger waits at **B**, and the destination is **R**.

## 1. Setup

`gym.make` wraps the core environment.
The model (`P`, `encode`, `decode`, ...) lives in the core `TaxiEnv`, which is `env.unwrapped`.

We build two versions:
- **dry** (`is_rainy=False`): deterministic;
- **rainy** (`is_rainy=True`): stochastic moves.

**What `fickle_passenger` means.** "Fickle" means changeable: a fickle passenger may change their mind about where they want to go.
- At `reset()`, the environment secretly decides, with probability `fickle_probability` (default $0.3$), whether this episode's passenger is fickle. The agent never observes this decision.
- If the passenger is fickle, then on the **first move the taxi makes with the passenger on board**, the destination $d$ is replaced by one of the other three landmarks, chosen uniformly at random.
- This happens at most once per episode. After it, the passenger keeps the new destination.

This idea comes from Dietterich's original Taxi problem (2000, Sec. 7.1).

Both versions use `fickle_passenger=False`, because the destination change is applied inside `step()` and is **not** written in the model `P`. With it turned on, `P` would describe a different problem from the one the environment simulates. Section 9 demonstrates this.

In [1]:
import numpy as np
import gymnasium as gym
from collections import Counter

print("gymnasium", gym.__version__)

env_dry   = gym.make("Taxi-v4", is_rainy=False, fickle_passenger=False, render_mode="ansi")
env_rainy = gym.make("Taxi-v4", is_rainy=True, rainy_probability=0.8,
                     fickle_passenger=False, render_mode="ansi")

u_dry, u_rainy = env_dry.unwrapped, env_rainy.unwrapped
print(env_dry)
print(type(u_dry))

gymnasium 1.4.0
<TimeLimit<OrderEnforcing<PassiveEnvChecker<TaxiEnv<Taxi-v4>>>>>
<class 'gymnasium.envs.toy_text.taxi.TaxiEnv'>


### The ANSI render

| Symbol | Meaning |
|---|---|
| yellow background | empty taxi |
| green background | taxi carrying the passenger |
| bold blue letter | passenger waiting at that landmark |
| magenta letter | destination |
| `\|` / `:` | wall / free passage |
| `(South)` etc. under the map | last action |

To render a chosen state, call `reset()` and then overwrite the internal state `env.unwrapped.s`.

We also give names to the actions and to the passenger/destination codes, so the code cells read like the hand calculations (for example `env.step(EAST)` instead of `env.step(2)`).

In [2]:
# actions a (index into the action space)
SOUTH, NORTH, EAST, WEST, PICKUP, DROPOFF = range(6)
ACTION_NAMES = ["South", "North", "East", "West", "Pickup", "Dropoff"]

# passenger location l and destination d (landmark indices; IN_TAXI only for l)
LOC_R, LOC_G, LOC_Y, LOC_B, IN_TAXI = range(5)
LOC_NAMES = ["R", "G", "Y", "B", "in taxi"]

def set_state(env, s):
    # Put the environment in state s. reset() first so the wrappers allow step().
    env.reset(seed=0)
    env.unwrapped.s = int(s)
    env.unwrapped.lastaction = None

def show(env, s=None):
    u = env.unwrapped
    s = u.s if s is None else s
    x, y, l, d = u.decode(s)
    print(f"s = {s}  ->  taxi=({x},{y}), passenger={LOC_NAMES[l]}, destination={LOC_NAMES[d]}")
    print(env.render())

set_state(env_dry, 252)
show(env_dry)

s = 252  ->  taxi=(2,2), passenger=B, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




## 2. State space $\mathcal S$

A state is $(x,y,\ell,d)$:
- $(x,y)\in\{0..4\}^2$ is the taxi's row and column;
- $\ell\in\{0..4\}$ is the passenger location: $0..3$ are the landmarks R, G, Y, B and $4$ means in the taxi;
- $d\in\{0..3\}$ is the destination.

**Expected size.** $|\mathcal S| = 5\cdot5\cdot5\cdot4 = 500$.

**Encoding** (mixed radix $5,5,5,4$):
$$
s = \big((5x+y)\,5+\ell\big)\,4+d .
$$

**Hand check.** $(2,2,3,0)\mapsto \big((5\cdot2+2)\cdot5+3\big)\cdot4+0 = 63\cdot4 = 252$.

In [3]:
nS = u_dry.observation_space.n
print("|S| =", nS)
assert nS == 500

def my_encode(x, y, l, d):
    return ((5*x + y)*5 + l)*4 + d

assert my_encode(2, 2, LOC_B, LOC_R) == u_dry.encode(2, 2, LOC_B, LOC_R) == 252
assert tuple(int(t) for t in u_dry.decode(252)) == (2, 2, LOC_B, LOC_R)

# my formula agrees with Gymnasium on all 500 states, and decode is its inverse
for s in range(nS):
    assert my_encode(*u_dry.decode(s)) == s
print("encode/decode verified on all 500 states")

|S| = 500
encode/decode verified on all 500 states


### Start states and reachable states

**Start states.** Episodes start with the passenger at a landmark ($\ell<4$) and a different destination ($\ell\ne d$).

**Expected:** $25\cdot4\cdot3 = 300$ start states.

**Reachable states.** From the start states you can reach:
- the 300 start states;
- the $25\cdot4 = 100$ states with the passenger in the taxi;
- the 4 states seen just after a successful drop-off. In these the taxi is at the destination and $\ell=d$: $(0,0,0,0)$, $(0,4,1,1)$, $(4,0,2,2)$, $(4,3,3,3)$.

**Expected:** $300+100+4=404$ reachable states. The other 96 are never visited.

In [4]:
start_states = [s for s in range(nS)
                if (lambda x, y, l, d: l != IN_TAXI and l != d)(*u_dry.decode(s))]
print("start states:", len(start_states))
assert len(start_states) == 300

# depth-first search over the model (a stack; the order does not change the set) (rainy has the larger support; both give the same set)
def reachable(u):
    seen, stack = set(start_states), list(start_states)
    while stack:
        s = stack.pop()
        for a in range(len(ACTION_NAMES)):
            for p, s2, r, t in u.P[s][a]:
                if p > 0 and s2 not in seen:
                    seen.add(s2)
                    if not t:
                        stack.append(s2)
    return seen

R_set = reachable(u_rainy)
post_terminal = sorted(s for s in R_set if (lambda x, y, l, d: l == d)(*u_dry.decode(s)))
print("reachable:", len(R_set), "| post-terminal:", post_terminal,
      [tuple(int(t) for t in u_dry.decode(s)) for s in post_terminal])
assert len(R_set) == 404
assert post_terminal == [my_encode(0, 0, LOC_R, LOC_R), my_encode(0, 4, LOC_G, LOC_G),
                         my_encode(4, 0, LOC_Y, LOC_Y), my_encode(4, 3, LOC_B, LOC_B)]

start states: 300
reachable: 404 | post-terminal: [0, 85, 410, 475] [(0, 0, 0, 0), (0, 4, 1, 1), (4, 0, 2, 2), (4, 3, 3, 3)]


## 3. Action space $\mathcal A$

$\mathcal A=\{0,\dots,5\}$, the same set in every state:

| $a$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| name | South $(x{+}1)$ | North $(x{-}1)$ | East $(y{+}1)$ | West $(y{-}1)$ | Pickup | Dropoff |

`action_mask(s)` marks the actions that **change** the state. It is a convenience, not part of the MDP.

**Expected for $s=252$** (empty taxi at $(2,2)$, no walls around it, passenger not here): the four moves are valid and Pickup/Dropoff are not. Mask $=[1,1,1,1,0,0]$.

In [5]:
nA = u_dry.action_space.n
print("|A| =", nA)
assert nA == 6
mask = u_dry.action_mask(252)
print("action_mask(252) =", mask)
# moves allowed, Pickup and Dropoff not
assert [ACTION_NAMES[a] for a in np.nonzero(mask)[0]] == ["South", "North", "East", "West"]
assert mask[PICKUP] == 0 and mask[DROPOFF] == 0

|A| = 6
action_mask(252) = [1 1 1 1 0 0]


## 4. Transition model $p(s'\mid s,a)$

### 4.1 The raw structure
`u.P[s][a]` is a list of tuples `(prob, next_state, reward, terminated)`.

**Expected for $s=252$ in the dry environment.** Each move goes to its neighbour cell with probability 1, and $\ell=3$, $d=0$ stay the same:

| $a$ | next cell | $s'$ | $r$ |
|---|---|---|---|
| South | $(3,2)$ | $((17)5+3)4 = 352$ | $-1$ |
| North | $(1,2)$ | $((7)5+3)4 = 152$ | $-1$ |
| East | $(2,3)$ | $((13)5+3)4 = 272$ | $-1$ |
| West | $(2,1)$ | $((11)5+3)4 = 232$ | $-1$ |
| Pickup | stay (passenger is at B) | $252$ | $-10$ |
| Dropoff | stay (passenger not in taxi) | $252$ | $-10$ |

In [6]:
for a in range(nA):
    print(f"{ACTION_NAMES[a]:8s}", u_dry.P[252][a])

expected = {SOUTH: (352, -1), NORTH: (152, -1), EAST: (272, -1), WEST: (232, -1),
            PICKUP: (252, -10), DROPOFF: (252, -10)}
for a, (s2, r) in expected.items():
    assert u_dry.P[252][a] == [(1.0, s2, r, False)]

South    [(1.0, 352, -1, False)]
North    [(1.0, 152, -1, False)]
East     [(1.0, 272, -1, False)]
West     [(1.0, 232, -1, False)]
Pickup   [(1.0, 252, -10, False)]
Dropoff  [(1.0, 252, -10, False)]


### 4.2 As tensors

Define
$$
P[a,s,s'] = p(s'\mid s,a),\qquad R[s,a]=r(s,a)=\sum_{s'} p(s'\mid s,a)\,r(s,a,s'),
$$
and the **terminating mass** $T[a,s,s']$, the part of $p$ whose tuple has `terminated=True`.

**Important:** accumulate with `+=`. In rainy mode one $(s,a)$ can list the same $s'$ more than once (Section 7).

**Expected checks:**
- every row $P[a,s,\cdot]$ sums to 1;
- the dry model has exactly one $s'$ per $(s,a)$, so it has $500\cdot6=3000$ nonzeros.

In [7]:
def mdp_arrays(u):
    nS, nA = u.observation_space.n, u.action_space.n
    P = np.zeros((nA, nS, nS))
    R = np.zeros((nS, nA))
    T = np.zeros((nA, nS, nS))
    for s in range(nS):
        for a in range(nA):
            for p, s2, r, t in u.P[s][a]:
                P[a, s, s2] += p
                R[s, a]     += p * r
                if t:
                    T[a, s, s2] += p
    return P, R, T

P, R, T = mdp_arrays(u_dry)
print("P shape", P.shape, "| R shape", R.shape)
print("row sums all 1:", np.allclose(P.sum(axis=2), 1.0))
print("nonzeros in P:", int((P > 0).sum()))
assert np.allclose(P.sum(axis=2), 1.0)
assert (P > 0).sum() == 3000

P shape (6, 500, 500) | R shape (500, 6)
row sums all 1: True
nonzeros in P: 3000


## 5. Rewards $r(s,a)$

| situation | $r$ |
|---|---|
| any move (even into a wall) | $-1$ |
| legal pickup | $-1$ |
| illegal pickup / illegal dropoff | $-10$ |
| dropoff at the destination with the passenger in the taxi | $+20$ (episode ends) |
| dropoff at **another** landmark with the passenger in the taxi | $-1$ (passenger is left there) |

**Expected values.** $R$ takes only the values $\{-10,-1,+20\}$. $+20$ appears exactly **4** times: one per landmark acting as destination.

**Hand-computed examples** (destination R, $d=0$):

| case | $s$ | $a$ | expected $(s', r, \text{term})$ |
|---|---|---|---|
| legal pickup at B | $(4,3,3,0)=472$ | Pickup | $(4,3,4,0)=476,\ -1,$ no |
| successful dropoff at R | $(0,0,4,0)=16$ | Dropoff | $(0,0,0,0)=0,\ +20,$ yes |
| dropoff at G (wrong landmark) | $(0,4,4,0)=96$ | Dropoff | $(0,4,1,0)=84,\ -1,$ no |
| East into a wall | $(0,1,3,0)=32$ | East | $32,\ -1,$ no |

In [8]:
print("distinct rewards:", np.unique(R))
assert set(np.unique(R)) == {-10.0, -1.0, 20.0}
assert (R == 20).sum() == 4

cases = [  # (s, a, expected (s', r, terminated))
    (472, PICKUP,  (476, -1, False)),
    ( 16, DROPOFF, (  0, 20, True)),
    ( 96, DROPOFF, ( 84, -1, False)),
    ( 32, EAST,    ( 32, -1, False)),
]
for s, a, exp in cases:
    (p, s2, r, t), = u_dry.P[s][a]
    print(f"s={s:3d} {u_dry.decode(s)}  {ACTION_NAMES[a]:7s} -> s'={s2:3d} {u_dry.decode(s2)}  r={r:+d}  term={t}")
    assert (s2, r, t) == exp

distinct rewards: [-10.  -1.  20.]
s=472 (4, 3, 3, 0)  Pickup  -> s'=476 (4, 3, 4, 0)  r=-1  term=False
s= 16 (0, 0, 4, 0)  Dropoff -> s'=  0 (0, 0, 0, 0)  r=+20  term=True
s= 96 (0, 4, 4, 0)  Dropoff -> s'= 84 (0, 4, 1, 0)  r=-1  term=False
s= 32 (0, 1, 3, 0)  East    -> s'= 32 (0, 1, 3, 0)  r=-1  term=False


### 5.1 Zhao's reward probability $p(r\mid s,a)$

**Why the book writes $p(r\mid s,a)$.** Zhao (Sec. 1.7) defines the model as two distributions:
- $p(s'\mid s,a)$: where you go next;
- $p(r\mid s,a)$: which reward you get.

In general the reward may be random, so it is described by a distribution. Both come from one joint distribution, which is the form Gymnasium stores in `P`:
$$
p(s'\mid s,a)=\sum_{r} p(s',r\mid s,a),\qquad
p(r\mid s,a)=\sum_{s'} p(s',r\mid s,a).
$$

**What the Bellman equation actually uses.** In Zhao's Eq. (2.7) the reward enters only through its mean:
$$
v_\pi(s)=\sum_a\pi(a\mid s)\Big[\underbrace{\sum_r p(r\mid s,a)\,r}_{=\,r(s,a)}+\gamma\sum_{s'}p(s'\mid s,a)\,v_\pi(s')\Big].
$$
So value and policy iteration only need $r(s,a)=\mathbb E[R\mid s,a]$, which is our `R[s,a]`.

**In Taxi-v4 the reward is deterministic given $(s,a)$.**
- Every tuple in `P[s][a]` carries the same reward, in both the dry and the rainy models (rain makes only $s'$ random; every move pays $-1$).
- Hence $p(r\mid s,a)$ puts probability $1$ on a single value, and $r(s,a)$ is that value.

**Expected for $s=252$** (dry or rainy):

| $a$ | $p(r\mid 252,a)$ | $r(252,a)$ |
|---|---|---|
| South, North, East, West | $p(-1)=1$ | $-1$ |
| Pickup, Dropoff | $p(-10)=1$ | $-10$ |

In [9]:
def reward_dist(u, s, a):
    # p(r|s,a) = sum over s' of the joint p(s', r|s,a) stored in P
    dist = Counter()
    for p, s2, r, t in u.P[s][a]:
        dist[r] += p
    return dict(dist)

for a in range(nA):
    print(f"{ACTION_NAMES[a]:8s} dry p(r|252,a) = {reward_dist(u_dry, 252, a)}   rainy = {reward_dist(u_rainy, 252, a)}")
assert all(reward_dist(u, 252, a) == ({-1: 1.0} if a in (SOUTH, NORTH, EAST, WEST) else {-10: 1.0})
           for u in (u_dry, u_rainy) for a in range(nA))

# deterministic reward for every (s,a): a single reward value in each p(r|s,a)
for u, Rm in [(u_dry, R), (u_rainy, mdp_arrays(u_rainy)[1])]:
    for s in range(nS):
        for a in range(nA):
            d = reward_dist(u, s, a)
            assert len(d) == 1 and np.isclose(sum(d.values()), 1.0)
            assert np.isclose(sum(r * q for r, q in d.items()), Rm[s, a])   # mean = r(s,a)
print("p(r|s,a) is a point mass for all 3000 pairs (dry and rainy), and its mean equals R[s,a]")

South    dry p(r|252,a) = {-1: 1.0}   rainy = {-1: 1.0}
North    dry p(r|252,a) = {-1: 1.0}   rainy = {-1: 1.0}
East     dry p(r|252,a) = {-1: 1.0}   rainy = {-1: 1.0}
West     dry p(r|252,a) = {-1: 1.0}   rainy = {-1: 1.0}
Pickup   dry p(r|252,a) = {-10: 1.0}   rainy = {-10: 1.0}
Dropoff  dry p(r|252,a) = {-10: 1.0}   rainy = {-10: 1.0}


p(r|s,a) is a point mass for all 3000 pairs (dry and rainy), and its mean equals R[s,a]


## 6. Termination and the start distribution $\mu_0$

### 6.1 Termination
The only terminating transitions are the 4 successful dropoffs. For dynamic programming, set the value after a terminating transition to zero by using the **continuation kernel**
$$
\tilde p(s'\mid s,a) = p(s'\mid s,a) - T[a,s,s'],
\qquad
v(s)=\max_a\Big[r(s,a)+\gamma\sum_{s'}\tilde p(s'\mid s,a)\,v(s')\Big].
$$

**Expected:**
- exactly 4 pairs $(s,a)$ have $\sum_{s'}\tilde p<1$;
- they are $a=5$ (Dropoff) at $s\in\{(0,0,4,0),(0,4,4,1),(4,0,4,2),(4,3,4,3)\}=\{16,97,418,479\}$.

In [10]:
Pc = P - T
a_idx, s_idx = np.nonzero(Pc.sum(axis=2) < 1)
pairs = sorted(zip(s_idx.tolist(), a_idx.tolist()))
print("terminating (s, a):", pairs, [tuple(int(t) for t in u_dry.decode(s)) for s, _ in pairs])
assert pairs == [(my_encode(0, 0, IN_TAXI, LOC_R), DROPOFF), (my_encode(0, 4, IN_TAXI, LOC_G), DROPOFF),
                 (my_encode(4, 0, IN_TAXI, LOC_Y), DROPOFF), (my_encode(4, 3, IN_TAXI, LOC_B), DROPOFF)]
assert [s for s, _ in pairs] == [16, 97, 418, 479]

terminating (s, a): [(16, 5), (97, 5), (418, 5), (479, 5)] [(0, 0, 4, 0), (0, 4, 4, 1), (4, 0, 4, 2), (4, 3, 4, 3)]


### 6.2 Start distribution $\mu_0$
**Expected:** uniform over the 300 start states, so $\mu_0(s)=1/300$ on its support. $s=252$ is a start state ($\ell=3\neq d=0$, $\ell<4$).

The discount $\gamma$ is **not** part of the environment; you choose it.

The 200-step `TimeLimit` is a wrapper. It is not part of the MDP.

In [11]:
mu0 = u_dry.initial_state_distrib
print("support:", int((mu0 > 0).sum()), "| values:", np.unique(mu0[mu0 > 0]), "| mu0(252) =", mu0[252])
assert (mu0 > 0).sum() == 300 and np.allclose(mu0[mu0 > 0], 1/300)
assert set(np.nonzero(mu0)[0]) == set(start_states)
print("TimeLimit max steps:", env_dry.spec.max_episode_steps)

support: 300 | values: [0.00333333] | mu0(252) = 0.0033333333333333335
TimeLimit max steps: 200


## 7. Rainy dynamics: a stochastic $p$

With `is_rainy=True` only the **moves** change. When the intended move is possible:
- the taxi goes forward with probability $q=0.8$;
- it slips to each side with probability $(1-q)/2 = 0.1$.

The sides depend on the heading:

| heading | left | right |
|---|---|---|
| South | East | West |
| North | West | East |
| East | North | South |
| West | South | North |

- A slip into a wall or the border leaves the taxi in place.
- If the intended move is blocked, the taxi stays with probability 1.
- Rewards stay at $-1$. Pickup and Dropoff stay deterministic.

**Expected, three cases** (passenger at B, destination R):

| case | $s$ | $a$ | expected $p(\cdot\mid s,a)$ |
|---|---|---|---|
| free cell | $252=(2,2)$ | East | $0.8\to272\,(2,3)$, $0.1\to152\,(1,2)$, $0.1\to352\,(3,2)$ |
| side slip blocked | $132=(1,1)$ | North | $0.8\to32\,(0,1)$, $0.1\to112\,(1,0)$, $0.1\to132$ (wall to the east) |
| forward blocked | $32=(0,1)$ | East | $1.0\to32$ |

The last case is stored as **three** identical tuples `(0.8,32), (0.1,32), (0.1,32)`, which is why we accumulate with `+=`.

In [12]:
Pr, Rr, Tr = mdp_arrays(u_rainy)
assert np.allclose(Pr.sum(axis=2), 1.0)

for s, a in [(252, EAST), (132, NORTH), (32, EAST)]:
    print(f"s={s} {u_rainy.decode(s)} {ACTION_NAMES[a]}: raw P = {[(round(p,2), s2) for p,s2,_,_ in u_rainy.P[s][a]]}")
    nz = np.nonzero(Pr[a, s])[0]
    print("        tensor p(.|s,a) =", {int(k): round(float(Pr[a, s, k]), 2) for k in nz})

assert np.isclose(Pr[EAST, 252, 272], 0.8) and np.isclose(Pr[EAST, 252, 152], 0.1) and np.isclose(Pr[EAST, 252, 352], 0.1)
assert np.isclose(Pr[NORTH, 132, 32], 0.8) and np.isclose(Pr[NORTH, 132, 112], 0.1) and np.isclose(Pr[NORTH, 132, 132], 0.1)
assert np.isclose(Pr[EAST, 32, 32], 1.0)

n_tuples = sum(len(u_rainy.P[s][a]) for s in range(nS) for a in range(nA))
print("raw tuples:", n_tuples, "| distinct (s,a,s'):", int((Pr > 0).sum()))
assert n_tuples == 500 * (4*3 + 2) == 7000

s=252 (2, 2, 3, 0) East: raw P = [(0.8, 272), (0.1, 152), (0.1, 352)]
        tensor p(.|s,a) = {152: 0.1, 272: 0.8, 352: 0.1}
s=132 (1, 1, 3, 0) North: raw P = [(0.8, 32), (0.1, 112), (0.1, 132)]
        tensor p(.|s,a) = {32: 0.8, 112: 0.1, 132: 0.1}
s=32 (0, 1, 3, 0) East: raw P = [(0.8, 32), (0.1, 32), (0.1, 32)]
        tensor p(.|s,a) = {32: 1.0}
raw tuples: 7000 | distinct (s,a,s'): 5660


### Empirical check of the model by sampling `step()`
We put the taxi in $s=252$ and press East $N=20\,000$ times, resetting the state each time.

**Expected frequencies:** $\approx 0.8,\ 0.1,\ 0.1$ for $272,\ 152,\ 352$.

The standard error is $\sqrt{0.1\cdot0.9/N}\approx 0.002$, so we allow a tolerance of $0.01$.

In [13]:
set_state(env_rainy, 252)
N, counts = 20_000, Counter()
for _ in range(N):
    u_rainy.s = 252
    s2, r, term, trunc, info = u_rainy.step(EAST)   # step the core env directly (no time limit)
    counts[s2] += 1
freq = {k: v / N for k, v in sorted(counts.items())}
print(freq)
for k, target in [(272, 0.8), (152, 0.1), (352, 0.1)]:
    assert abs(freq[k] - target) < 0.01

{152: 0.1031, 272: 0.79885, 352: 0.09805}


## 8. Manual interaction: playing an episode by hand

We start the dry environment at $s=252$ (empty taxi at $(2,2)$, passenger at B, destination R) and follow the shortest plan: 3 moves to B, Pickup, 7 moves to R around the walls, Dropoff.

**Expected trajectory** (each row decodes $s_t=(x,y,\ell,d)$; the next row starts from $s_{t+1}$):

| $t$ | $s_t$ | taxi $(x,y)$ | passenger $\ell$ | destination $d$ | action $a_t$ | reward $r_{t+1}$ | $s_{t+1}$ |
|---|---|---|---|---|---|---|---|
| 0 | 252 | $(2,2)$ | B | R | East (to B) | $-1$ | 272 |
| 1 | 272 | $(2,3)$ | B | R | South (to B) | $-1$ | 372 |
| 2 | 372 | $(3,3)$ | B | R | South (to B) | $-1$ | 472 |
| 3 | 472 | $(4,3)$ | B | R | Pickup (at B) | $-1$ | 476 |
| 4 | 476 | $(4,3)$ | taxi | R | North (to R) | $-1$ | 376 |
| 5 | 376 | $(3,3)$ | taxi | R | North (to R) | $-1$ | 276 |
| 6 | 276 | $(2,3)$ | taxi | R | West (to R) | $-1$ | 256 |
| 7 | 256 | $(2,2)$ | taxi | R | West (to R) | $-1$ | 236 |
| 8 | 236 | $(2,1)$ | taxi | R | North (to R) | $-1$ | 136 |
| 9 | 136 | $(1,1)$ | taxi | R | North (to R) | $-1$ | 36 |
| 10 | 36 | $(0,1)$ | taxi | R | West (to R) | $-1$ | 16 |
| 11 | 16 | $(0,0)$ | taxi | R | Dropoff (at R) | $+20$ | 0 |
| 12 | 0 | $(0,0)$ | R | R | -- (terminal) | | |

**Expected rewards:** eleven $-1$ (10 moves and the pickup), then $+20$ at step 12, which terminates the episode. The undiscounted return is $9$.

With $\gamma=0.9$:
$$
G_0=-\sum_{k=0}^{10}0.9^k+20\cdot0.9^{11}
=-\frac{1-0.9^{11}}{0.1}+20\cdot0.9^{11}\approx-0.5857 .
$$

In [14]:
plan = [EAST, SOUTH, SOUTH, PICKUP,                     # go to B and pick up
        NORTH, NORTH, WEST, WEST, NORTH, NORTH, WEST,     # go to R around the walls
        DROPOFF]
expected_states = [272, 372, 472, 476, 376, 276, 256, 236, 136, 36, 16, 0]

set_state(env_dry, 252)
rewards, states = [], []
for k, a in enumerate(plan):
    s2, r, term, trunc, info = env_dry.step(a)
    rewards.append(r); states.append(s2)
    print(f"step {k+1:2d}: {ACTION_NAMES[a]:7s} r={r:+3d} term={term}")
    show(env_dry)

assert states == expected_states
assert rewards == [-1]*11 + [20] and term
gamma = 0.9
G = sum(gamma**k * r for k, r in enumerate(rewards))
G_hand = -(1 - gamma**11) / (1 - gamma) + 20 * gamma**11
print(f"undiscounted return = {sum(rewards)},  G_0 = {G:.4f},  hand formula = {G_hand:.4f}")
assert sum(rewards) == 9 and np.isclose(G, G_hand) and np.isclose(G, -0.5857, atol=1e-4)

step  1: East    r= -1 term=False
s = 272  ->  taxi=(2,3), passenger=B, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (East)

step  2: South   r= -1 term=False
s = 372  ->  taxi=(3,3), passenger=B, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)

step  3: South   r= -1 term=False
s = 472  ->  taxi=(4,3), passenger=B, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)

step  4: Pickup  r= -1 term=False
s = 476  ->  taxi=(4,3), passenger=in taxi, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Pickup)

step  5: North   r= -1 term=False
s = 376  ->  taxi=(3,3), passenger=in taxi, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : |_: |
|Y| : |B: |
+---------+
  (North)

step  6: North   r= -1 term=False
s = 276  ->  taxi=(2,3), passenger=in taxi, desti

### Three "mistakes" that show the dynamics
From $s=252$ in the dry environment:
1. **Pickup when the passenger is elsewhere:** $s'=252$, $r=-10$.
2. **Moving into a wall:** go North twice to $(0,2)$, then West. The wall `|` between columns 1 and 2 in row 0 blocks the move, so $s'=s$ and $r=-1$.
3. **Dropoff with an empty taxi:** $r=-10$, and the state does not change.

In [15]:
set_state(env_dry, 252)
s2, r, *_ = env_dry.step(PICKUP); print("Pickup  ->", s2, r); assert (s2, r) == (252, -10)

env_dry.step(NORTH); s_top, _, *_ = env_dry.step(NORTH)            # now at (0,2)
assert tuple(int(t) for t in u_dry.decode(s_top))[:2] == (0, 2)
s2, r, *_ = env_dry.step(WEST); print("West at (0,2) ->", s2, r); assert (s2, r) == (s_top, -1)
show(env_dry)

s2, r, *_ = env_dry.step(DROPOFF); print("Dropoff (empty) ->", s2, r); assert (s2, r) == (s_top, -10)

Pickup  -> 252 -10
West at (0,2) -> 52 -1
s = 52  ->  taxi=(0,2), passenger=B, destination=R
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (West)

Dropoff (empty) -> 52 -10


## 9. Why `fickle_passenger` is not in the model

With `fickle_passenger=True`:
- `reset()` sets a **hidden** flag $f=1$ with probability $0.3$;
- on the first move made while the passenger is in the taxi, if $f=1$, `step()` replaces the destination with one of the other three landmarks.

This happens **after** sampling from `P`, so `P` never shows it.

**Expected demonstration.**
- Take $s=476=(4,3,4,0)$: the passenger was just picked up at B, destination R.
- Force $f=1$ and press North.
- The model says $p(376\mid 476,\text{North})=1$.
- The environment returns $(3,3,4,d')$ with $d'\neq0$, a state the model gives probability $0$.

So for dynamic programming keep `fickle_passenger=False`. The exact model with fickleness is an augmented MDP on $(s,f)$ with 1000 states.

In [16]:
env_f = gym.make("Taxi-v4", fickle_passenger=True, render_mode="ansi")
u_f = env_f.unwrapped
env_f.reset(seed=0)
u_f.s, u_f.fickle_step = 476, True        # force the hidden flag

model_support = {s2 for p, s2, _, _ in u_f.P[476][NORTH] if p > 0}
s2, r, *_ = env_f.step(NORTH)
print("model support:", model_support, "| observed s' =", s2, u_f.decode(s2))
show(env_f)
assert s2 not in model_support and u_f.decode(s2)[3] != LOC_R

model support: {376} | observed s' = 377 (3, 3, 4, 1)
s = 377  ->  taxi=(3,3), passenger=in taxi, destination=G
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : |_: |
|Y| : |B: |
+---------+
  (North)



## 10. Summary: the extracted MDP

| component | object | value / shape |
|---|---|---|
| $\mathcal S$ | `range(500)`, `encode`/`decode` | 500 states (300 start, 404 reachable) |
| $\mathcal A$ | `range(6)` | S, N, E, W, Pickup, Dropoff |
| $p(s'\mid s,a)$ | `P[a,s,s']` | $(6,500,500)$; deterministic (dry) or $0.8/0.1/0.1$ (rainy) |
| $r(s,a)$ | `R[s,a]` | values in $\{-10,-1,+20\}$; $p(r\mid s,a)$ is a point mass at $r(s,a)$ |
| termination | `T`, $\tilde p = P-T$ | 4 pairs (Dropoff at the destination) |
| $\mu_0$ | `initial_state_distrib` | uniform on 300 states |
| $\gamma$ | your choice | not set by the environment |

These arrays (`P`, `R`, `Pc`, `mu0`) are the inputs to value iteration.